In [40]:
import json
from datetime import datetime
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.agents import create_agent
from langchain_community.tools import GoogleSerperResults,TavilySearchResults
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
# llm = ChatOllama(model="gemma4:31b-cloud")
# llm = ChatOpenAI(model='gpt-5.4')

# search_tool = GoogleSerperResults()
search_tool = TavilySearchResults(search_depth='basic')

@tool
def get_system_time(format:str='%Y-%m-%d %H:%M:%S'):
    """
    return the current date and time
    """
    current_date = datetime.now()
    format_time = current_date.strftime(format)
    return format_time


def _banner(title: str) -> None:
    line = "=" * 56
    print(f"\n{line}\n {title}\n{line}")

def _truncate(s: str, max_chars: int) -> str:
    return s if len(s) <= max_chars else s[: max_chars - 3] + "..."

def _reasoning(msg: AIMessage) -> str | None:
    raw = (msg.additional_kwargs or {}).get("reasoning_content")
    if isinstance(raw, str) and raw.strip():
        return raw.strip()
    return None

def print_react_update(chunk: dict, observation_max_chars: int = 1800) -> None:
    for node_name, payload in chunk.items():
        if node_name == "model":
            for msg in payload.get("messages", []):
                if not isinstance(msg, AIMessage):
                    continue
                if (r := _reasoning(msg)):
                    _banner("Thinking")
                    print(r)
                if msg.tool_calls:
                    _banner("Action")
                    for tc in msg.tool_calls:
                        print(f"  tool: {tc.get('name', '')}")
                        print(f"  input: {json.dumps(tc.get('args', {}), ensure_ascii=False)}")
                text = (msg.content or "").strip()
                if text and not msg.tool_calls:
                    _banner("Final answer")
                    print(text)
                elif text and msg.tool_calls:
                    _banner("Assistant (partial text)")
                    print(text)
        elif node_name == "tools":
            for msg in payload.get("messages", []):
                if isinstance(msg, ToolMessage):
                    _banner("Observation (tool output)")
                    print(_truncate(str(msg.content), observation_max_chars))

SYSTEM = """You answer using web search when the user asks for current facts, news, weather, prices, or anything time-sensitive.
- Call the tool `google_serper_results_json` with one concise search query (or a second query if the first results are weak).
- Ground every factual claim in the JSON you received: paraphrase snippets and include source titles and URLs when present.
- If the tool returns no useful evidence, say you could not verify and avoid inventing details."""

agent = create_agent(llm, tools=[search_tool,get_system_time], system_prompt=SYSTEM)



In [43]:


question = "What is the weather in Hyderabad today? One short line with a source."
# question = "when was the spaceX last lunch and how many days ago was that from this instant"

state = {"messages": [HumanMessage(content=question)]}

print("User:", question)
for step in agent.stream(state, stream_mode="updates"):
    print_react_update(step)


User: What is the weather in Hyderabad today? One short line with a source.

 Thinking
We need current weather. Must use web search. Use tavily_search_results_json.

 Action
  tool: tavily_search_results_json
  input: {"query": "Hyderabad weather today"}

 Observation (tool output)
[{"title": "Weather in Hyderabad", "url": "https://www.weatherapi.com/", "content": "{'location': {'name': 'Hyderabad', 'region': 'Telangana', 'country': 'India', 'lat': 17.3753, 'lon': 78.4744, 'tz_id': 'Asia/Kolkata', 'localtime_epoch': 1778788777, 'localtime': '2026-05-15 01:29'}, 'current': {'last_updated_epoch': 1778787900, 'last_updated': '2026-05-15 01:15', 'temp_c': 31.4, 'temp_f': 88.6, 'is_day': 0, 'condition': {'text': 'Thundery outbreaks in nearby', 'icon': '//cdn.weatherapi.com/weather/64x64/night/200.png', 'code': 1087}, 'wind_mph': 8.7, 'wind_kph': 14.0, 'wind_degree': 162, 'wind_dir': 'SSE', 'pressure_mb': 1002.0, 'pressure_in': 29.6, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 51, 'cloud